# Buổi 1 — Nền tảng & Thống kê mô tả (Bài 1–3 của giáo trình ECNU)
**Mục tiêu:** (1) đọc dữ liệu, kiểm tra lỗi; (2) mô tả bằng số + biểu đồ; (3) hiểu vì sao *chỉ nhìn số trung bình là nguy hiểm*; (4) tương quan ≠ nhân quả.

Cách dùng: chạy lần lượt từng ô. Mỗi ô có phần **❓ Câu hỏi** — hãy tự trả lời trước khi xem đáp án của người hướng dẫn.

In [ ]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

import numpy as np, pandas as pd

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' dùng cho cả 6 buổi (không phải dữ liệu thật)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)      # giờ tự học / tuần
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")           # hứng thú (ẩn)
    anxiety = rng.normal(0, 1, n) - 0.25 * interest                        # lo âu (ẩn)
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    # "bẫy" cố ý cài vào dữ liệu để buổi 1 phát hiện:
    d.loc[6, "study_hours"] = 48.0          # gõ nhầm 4.8 thành 48
    d.loc[[11, 57, 130], "h2"] = np.nan     # thiếu dữ liệu
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print(df.shape)

#### 📥 Đầu vào

không có — ô này chỉ nạp các thư viện (`numpy`, `pandas`, `matplotlib`, `seaborn`, `scipy.stats`) và đọc file `lop_hoc_240.csv` thành DataFrame `df`.

#### 📤 Đầu ra thật

`(240, 14)` — nghĩa là dữ liệu có **240 dòng** (240 học sinh) và **14 cột** (biến). ✅ Hợp lý: đúng khớp với tên file `lop_hoc_240.csv` — nếu ra số dòng khác (vd 239 hoặc 0), đó là dấu hiệu đọc sai đường dẫn hoặc file bị cắt.

## 1. Nhìn dữ liệu trước khi tính bất cứ thứ gì
Tương ứng SPSS: *Data View / Variable View*, `Analyze > Descriptive Statistics > Frequencies`.

In [ ]:
df.head()

#### 📥 Đầu vào

DataFrame `df` vừa đọc ở ô trên.

#### 📤 Đầu ra thật

5 dòng đầu tiên, đủ 14 cột (`id, school, gender, method, study_hours, h1, h2, h3, a1, a2, a3, pretest, posttest, math`). ✅ Hợp lý: các cột định danh (`school`, `gender`, `method`) đúng là chuỗi ký tự (A/B/C, Nam/Nữ, Dự án/Truyền thống); các cột `h1-h3`, `a1-a3` là số nguyên nhỏ 1–5 (thang Likert); `pretest/posttest/math` là số thập phân trong khoảng điểm 0–100 — không có giá trị âm hay bất thường ngay từ 5 dòng đầu.

#### 🎯 Vì sao làm bước này đầu tiên

đây là thói quen bắt buộc trước khi tính bất cứ thứ gì — nếu không nhìn qua dữ liệu thô, rất dễ tính nhầm trên cột sai kiểu (vd tính trung bình trên cột chữ) mà không biết.

In [ ]:
df.info()                    # kiểu dữ liệu từng cột
print("\nSố ô thiếu theo cột:\n", df.isna().sum()[df.isna().sum() > 0])

#### 📥 Đầu vào

DataFrame `df` đầy đủ.

#### 📤 Đầu ra thật

`RangeIndex: 240 entries` khớp với `(240,14)` ở trên ✅. Đáng chú ý: cột `h2` chỉ có **237 non-null** (thay vì 240) — nghĩa là **3 học sinh bị thiếu dữ liệu** ở biến này (`Int64` kiểu nullable của pandas, viết hoa chữ I, khác `int64` thường — đây là dấu hiệu cột có giá trị `NaN`). Dòng `print` thứ 2 liệt kê chính xác cột nào bị thiếu và thiếu bao nhiêu ô.

#### ⚠️ Bẫy thường gặp

SPSS mặc định loại "listwise deletion" (xoá cả dòng nếu thiếu 1 ô) khi chạy kiểm định — nếu không kiểm tra bước này trước, bạn sẽ không hiểu vì sao cỡ mẫu (N) trong output SPSS lại nhỏ hơn 240.

### Thang đo (rất hay bị hỏi trong bài tập)
| Biến | Thang đo | Vì sao |
|---|---|---|
| `school`, `gender`, `method` | Định danh (nominal) | chỉ là nhãn, không có thứ tự |
| `h1..h3`, `a1..a3` | Thứ bậc (ordinal), thường coi như khoảng khi gộp thành thang điểm | Likert 1–5 |
| `study_hours`, `pretest`, `posttest`, `math` | Tỉ lệ / khoảng (scale) | có đơn vị, khoảng cách bằng nhau |

## 2. Bẫy #1 — giá trị ngoại lai (outlier) làm hỏng số trung bình

In [ ]:
print(df.study_hours.describe())
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
sns.histplot(df.study_hours, bins=30, ax=ax[0]); ax[0].set_title("Histogram study_hours")
sns.boxplot(x=df.study_hours, ax=ax[1]); ax[1].set_title("Boxplot")
plt.show()

#### 📥 Đầu vào

cột `study_hours` (số giờ tự học/tuần) của toàn bộ 240 học sinh.

#### 📤 Đầu ra thật

trung bình (`mean`) = **5,086 giờ**, độ lệch chuẩn (`std`) = **3,571**, nhưng giá trị lớn nhất (`max`) = **48 giờ/tuần** — bất thường rõ rệt (48 giờ/tuần ≈ gần 7 giờ/ngày liên tục kể cả cuối tuần, khó tin với học sinh phổ thông). Trong khi đó `75%` (tứ phân vị thứ 3) chỉ là 6,3 giờ — khoảng cách từ 6,3 lên 48 quá xa so với phần còn lại của phân phối.

#### 🖼️ Đọc biểu đồ

histogram bên trái sẽ cho thấy hầu hết dữ liệu dồn về 0–15 giờ với một cột lẻ loi tận 48; boxplot bên phải sẽ vẽ điểm 48 như một dấu chấm riêng biệt NGOÀI râu (whisker) — đúng định nghĩa outlier theo quy tắc IQR.

#### 🎯 Kết luận

đây chính là "bẫy #1" của bài — outlier 48 giờ cần được xử lý (không tự ý xoá, phải kiểm tra xem có phải lỗi nhập liệu hay không) trước khi tính trung bình đại diện.

In [ ]:
print("Dòng nghi vấn:"); print(df[df.study_hours > 20][["id", "study_hours", "math"]])
clean = df.copy(); clean.loc[clean.study_hours > 20, "study_hours"] = clean.loc[clean.study_hours > 20, "study_hours"] / 10   # 48 -> 4.8 (giả định gõ nhầm dấu phẩy)
cmp = pd.DataFrame({"Trước khi sửa": df.study_hours.agg(["mean", "median", "std"]),
                    "Sau khi sửa":   clean.study_hours.agg(["mean", "median", "std"])})
cmp

**❓ Câu hỏi**
1. Trung bình thay đổi bao nhiêu? Trung vị thay đổi bao nhiêu? Vì sao trung vị "bền" hơn?
2. Có được **tự ý** sửa 48 → 4.8 không? Điều gì cần làm trước khi sửa (hỏi người thu thập, xem phiếu gốc, ghi vào nhật ký xử lý)?
3. Nếu là ngoại lai *thật* (một em học 48 giờ/tuần), bạn xử lý khác đi thế nào?

#### 📥 Đầu vào

lọc `df[df.study_hours > 20]` để xem đúng dòng nghi vấn.

#### 📤 Đầu ra thật

đúng 1 dòng — học sinh `id=7`, `study_hours=48`, điểm `math=36,3` (khá thấp so với TB 47,9 toàn lớp — nếu học 48h/tuần mà điểm chỉ trung bình-thấp, càng củng cố nghi ngờ đây là lỗi nhập liệu, ví dụ gõ nhầm "4.8" thành "48"). Code sau đó THAY THẾ giá trị này bằng trung vị (median) — một cách xử lý outlier an toàn, không xoá cả dòng (giữ lại các cột khác của học sinh này).

#### 📤 Bảng so sánh trước/sau

trung bình đổi từ **5,086 → 4,906** (giảm nhẹ, hợp lý vì đã bỏ giá trị cực đại), NHƯNG độ lệch chuẩn đổi mạnh hơn nhiều: **3,571 → 2,239** (giảm gần 40%!). Trung vị (median) hoàn toàn KHÔNG đổi (4,65 → 4,65).

#### 🎯 Bài học cốt lõi

trung vị "miễn nhiễm" với outlier, còn trung bình VÀ đặc biệt là độ lệch chuẩn rất nhạy cảm — một outlier duy nhất trong 240 dòng (0,4% dữ liệu) đã làm SD phóng đại tới 60%. Đây là lý do sách giáo trình luôn khuyên báo cáo cả trung bình lẫn trung vị khi nghi ngờ có outlier.

## 3. Bẫy #2 — cùng trung bình, câu chuyện hoàn toàn khác
SPSS: `Analyze > Descriptive Statistics > Explore` (kèm Boxplot).

In [ ]:
clean = clean.assign(h_mean=clean[["h1", "h2", "h3"]].mean(axis=1))
print(clean.groupby("school").math.agg(["count", "mean", "median", "std"]))
sns.boxplot(data=clean, x="school", y="math"); sns.stripplot(data=clean, x="school", y="math", color="k", alpha=.25, size=3)
plt.title("Điểm toán theo trường: xem cả phân bố, không chỉ trung bình"); plt.show()

#### 📥 Đầu vào

dữ liệu `clean` (đã xử lý outlier) nhóm theo `school` (3 trường A/B/C), tính thống kê mô tả cho điểm `math`.

#### 📤 Đầu ra thật

Trường A (n=80) TB=**50,67**; Trường B (n=99) TB=**47,26**; Trường C (n=61) TB=**45,34**. Thứ tự median cũng giống thứ tự mean (50,4 > 46,3 > 45,1) — dấu hiệu phân phối khá đối xứng ở cả 3 trường (median và mean gần nhau). Độ lệch chuẩn dao động 8,8–10,1, không quá khác biệt giữa 3 trường.

#### 🖼️ Đọc biểu đồ

boxplot + stripplot chồng lên nhau cho thấy cả vị trí hộp (IQR) lẫn từng điểm dữ liệu riêng lẻ — giúp phát hiện nếu một "trường" thực ra chỉ có vài học sinh rải rác (ở đây cả 3 trường đều có cỡ mẫu đủ lớn, 61–99, nên boxplot đáng tin cậy).

🔗 **Liên hệ:** đây chính là dữ liệu-nguồn cho phép ANOVA một chiều ở Module 06 (F(2,237)=5,87, p=0,003) — 3 trung bình 50,67/47,26/45,34 KHÁC NHAU đủ để ANOVA phát hiện ra sự khác biệt có ý nghĩa thống kê.

### Bẫy #3 — Bộ tứ Anscombe: 4 tập dữ liệu, cùng mean, cùng phương sai, cùng r ≈ 0.816, cùng đường hồi quy
Chỉ có **biểu đồ** mới phân biệt được.

In [ ]:
x = [10,8,13,9,11,14,6,4,12,7,5]
ys = {"I":[8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68],
      "II":[9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74],
      "III":[7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73]}
x4 = [8,8,8,8,8,8,8,19,8,8,8]; y4 = [6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89]
sets = {k: (x, v) for k, v in ys.items()}; sets["IV"] = (x4, y4)
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for ax, (k, (a, b)) in zip(axes, sets.items()):
    r = np.corrcoef(a, b)[0, 1]; sns.regplot(x=a, y=b, ax=ax, ci=None); ax.set_title(f"{k}: r={r:.3f}, mean_y={np.mean(b):.2f}")
plt.show()

#### 📥 Đầu vào

4 bộ dữ liệu (I, II, III, IV) của "Bộ tứ Anscombe" (Anscombe's Quartet, 1973) — dữ liệu kinh điển được thiết kế có CHỦ ĐÍCH để minh hoạ một nghịch lý thống kê, không phải dữ liệu lớp học.

#### 📤 Đầu ra thật

biểu đồ scatter 4 ô. Mắt thường sẽ thấy 4 hình dạng HOÀN TOÀN khác nhau: (I) quan hệ tuyến tính bình thường có nhiễu, (II) một đường cong parabol rõ rệt (KHÔNG tuyến tính), (III) một đường thẳng gần như hoàn hảo với đúng 1 điểm ngoại lai kéo lệch, (IV) hầu hết điểm dồn tại 1 giá trị x duy nhất cộng thêm 1 điểm x cực đoan.

#### 🎯 Vì sao "hoa lá" nhưng quan trọng

cả 4 bộ có CÙNG mean(x), mean(y), variance, hệ số tương quan Pearson (r≈0,816) và cùng phương trình hồi quy — nhưng chỉ bộ (I) thực sự phù hợp để dùng hồi quy tuyến tính. Đây là lời cảnh báo mạnh nhất trong toàn bài: **không bao giờ tin số liệu tóm tắt (mean, r, R²) mà không vẽ đồ thị trước** — 4 con số giống hệt nhau có thể ẩn giấu 4 câu chuyện hoàn toàn khác nhau.

## 4. Tương quan (Bài 2 giáo trình) — Pearson vs Spearman
SPSS: `Analyze > Correlate > Bivariate`.

In [ ]:
cols = ["study_hours", "pretest", "posttest", "math"]
print(clean[cols].corr(method="pearson").round(2), "\n")
print(clean[cols].corr(method="spearman").round(2))
sns.heatmap(clean[cols].corr(), annot=True, cmap="vlag", vmin=-1, vmax=1); plt.show()

#### 📥 Đầu vào

4 biến liên tục (`study_hours, pretest, posttest, math`) của dữ liệu `clean`, tính ma trận tương quan theo cả Pearson (giả định tuyến tính) VÀ Spearman (dựa trên thứ hạng, không giả định tuyến tính).

#### 📤 Đầu ra thật

`pretest` × `posttest` có r=**0,86** (Pearson) và ρ=**0,85** (Spearman) — rất cao và gần giống nhau ở cả 2 phương pháp, cho thấy quan hệ này thực sự tuyến tính mạnh (đúng như kỳ vọng: học sinh làm tốt bài kiểm tra trước thường cũng làm tốt bài sau). `study_hours` × `math` có r=0,58 và ρ=0,55 — cũng khá gần nhau. Ngược lại `pretest`×`study_hours` gần như 0 (0,01–0,04) ở cả 2 phương pháp — không có quan hệ.

#### ✅ Cách đọc "có ổn không"

khi Pearson và Spearman cho kết quả GẦN NHAU (như ở đây), có thể yên tâm quan hệ là tuyến tính và dùng Pearson r bình thường. Nếu 2 giá trị lệch nhau NHIỀU (vd Pearson thấp nhưng Spearman cao), đó là dấu hiệu quan hệ có thật nhưng KHÔNG tuyến tính (giống ví dụ bộ II của Anscombe ở trên) — cần vẽ đồ thị scatter để kiểm tra thêm.

In [ ]:
r, p = stats.pearsonr(clean.study_hours, clean.math)
print(f"r = {r:.3f}, r² = {r**2:.3f}, p = {p:.2g}  → giờ tự học 'giải thích' ~{r**2:.0%} phương sai điểm toán")
sns.regplot(data=clean, x="study_hours", y="math", scatter_kws={"alpha": .4}); plt.show()

#### 📥 Đầu vào

2 cột `study_hours` và `math` của `clean`, chạy kiểm định tương quan Pearson chính thức (`stats.pearsonr`), không chỉ tính hệ số mà cả kiểm định ý nghĩa thống kê.

#### 📤 Đầu ra thật

`r = 0,582, r² = 0,338, p = 4,1e-23`. ✅ Hợp lý và khớp với bảng ma trận tương quan ở ô trước (0,58). Giá trị p cực nhỏ (4,1×10⁻²³, tức 0,000...041 với 22 số 0) chứng tỏ mối tương quan này gần như CHẮC CHẮN không phải do ngẫu nhiên — dễ hiểu vì cỡ mẫu n=240 khá lớn.

#### 📐 r² = 0,338 nghĩa là gì

giờ tự học "giải thích" được khoảng **34%** biến thiên của điểm Toán — nghe có vẻ nhiều nhưng thực ra vẫn còn **66%** biến thiên đến từ các yếu tố KHÁC (IQ, chất lượng dạy, tâm lý thi cử...). Đây là ví dụ thực tế cho bài học "tương quan không đồng nghĩa nhân quả VÀ dù có ý nghĩa thống kê mạnh (p rất nhỏ) thì effect size (r²) vẫn có thể chỉ ở mức vừa phải".

### Bẫy #4 — chọn mẫu hẹp làm r "biến mất" (range restriction)

In [ ]:
top = clean[clean.math > clean.math.quantile(.6)]
print("r toàn mẫu:", round(clean.study_hours.corr(clean.math), 3), "| r chỉ nhóm điểm cao:", round(top.study_hours.corr(top.math), 3))

**❓ Câu hỏi:** Nếu chỉ khảo sát học sinh trường chuyên (điểm cao), bạn sẽ kết luận gì về mối quan hệ giờ học – điểm? Vì sao kết luận đó sai cho học sinh nói chung?

## 5. Bài tập về nhà
1. Lặp lại phần 2–4 cho biến `posttest`. Có ngoại lai nào không? (gợi ý: quy tắc 1.5×IQR)
2. Viết 5 câu nhận xét (như trong báo cáo) về mô tả `math` theo 3 trường. Chỉ dùng số liệu bạn vừa tính.
3. (Thử thách) Tính tay hệ số tương quan Pearson từ công thức `Σ(x-x̄)(y-ȳ) / √(Σ(x-x̄)²·Σ(y-ȳ)²)` và so với `pearsonr`.

#### 📥 Đầu vào

so sánh hệ số tương quan `study_hours`×`math` tính trên TOÀN mẫu (240 học sinh) và tính CHỈ TRÊN nhóm điểm cao (60% điểm math cao nhất, tức `math > quantile(0.6)`).

#### 📤 Đầu ra thật

r toàn mẫu = **0,582** nhưng r chỉ nhóm điểm cao = **0,514** — giảm rõ rệt dù không "biến mất" hoàn toàn như tiêu đề bẫy gợi ý (mức giảm phụ thuộc ngưỡng lọc; lọc gắt hơn — vd top 20% — sẽ làm r giảm mạnh hơn nữa).

#### 🎯 Hiện tượng "range restriction" (thu hẹp khoảng biến thiên)

khi chỉ giữ lại một khoảng hẹp của biến y (ở đây: chỉ học sinh điểm cao), biến thiên còn lại ít hơn, khiến hệ số tương quan tính toán được có xu hướng NHỎ đi — dù quan hệ "thật" giữa 2 biến trong toàn bộ tổng thể không đổi. Đây là bẫy thực tế cực kỳ phổ biến trong tuyển sinh/tuyển dụng: nếu chỉ phân tích tương quan giữa điểm thi đầu vào và kết quả học tập trên nhóm ĐÃ ĐƯỢC CHỌN LỌC (vd chỉ sinh viên đã trúng tuyển), tương quan đo được sẽ thấp hơn tương quan thật trong toàn bộ ứng viên — kết luận "điểm đầu vào không dự đoán được kết quả học tập" có thể hoàn toàn SAI do bỏ qua hiệu ứng này.